In [ ]:
!python --version

## Monai wiht EfficientNet-B0/B3 (lightweight, great for cell morphology)

![BO](images/EfficientNet-Architecture-diagram.png)

<br>

![B3](images/Architecture-of-EfficientNet-B0-with-MBConv-as-Basic-building-blocks.png)


### EfficientNet is lighter, faster, and more accurate than DenseNet for microscopy.

| Model | Input Size | Parameters | Notes |
|-------|-------|-------|-------|
|B0 | 224×224 | 5.3M  | fast, small GPU |
| B3 | 300×300 | 12M | higher accuracy for cellular phenotypes |

In [ ]:
!nvidia-smi

In [ ]:
import os, sys
import numpy as np

# import pandas as pd
# from glob import glob
# from tqdm import tqdm

import matplotlib.pyplot as plt

from pathlib import Path
ROOT = Path().resolve().parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.Basic import *
from libs.image_lib import *
from libs.neural_network_lib import *

import numpy
numpy.__version__

In [ ]:
!nvidia-smi

In [ ]:
import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torch.optim as optim
# from torch.optim.lr_scheduler import ExponentialLR
# from torch.utils.data import Dataset, DataLoader, Subset

print(">> Cuda Ok?", torch.cuda.is_available())

if torch.cuda.is_available():
    print("\t>> device counts:", torch.cuda.device_count())
    print("\t>> current device:", torch.cuda.current_device())
    print("\t>> device name:", torch.cuda.get_device_name(0))

print(">> torch version:", torch.__version__)

In [ ]:
import monai
monai.__version__

In [ ]:
root_data = "../data"
os.listdir(root_data)

In [ ]:
cp = Cellpose(root0_data=root_data, verbose=True)

In [ ]:
cp.set_default_parameters(root_yaml='..', verbose=True)

In [ ]:
cp.root_samples, cp.root_crop

In [ ]:
plates = cp.list_plates(s_start='Plate')
plates

In [ ]:
i=0
plate=plates[i]
print(">>> plate", plate)

cp.set_plate_params(plate=plate, verbose=True)

In [ ]:
cp.experiments

In [ ]:
cp.set_plate_params(plate=plate, verbose=False)

ncrop=5

for experiment in cp.experiments:
    cp.create_roots_experiment(experiment, verbose=False)
    fname_imgs = cp.list_crop_images_already_set(ncrop=ncrop, image_type='png', verbose=False)

    key = f"{plate} - {experiment}"
    print(key, len(fname_imgs), cp.root_crop_image)

In [ ]:
filename = os.path.join(cp.root_crop_image, fname_imgs[0])
os.path.exists(filename), filename

### Create dataset dictionary list

In [ ]:
train_list, test_list, dft = cp.create_train_and_test_dataset(ncrop=5, perc_train=0.6, perc_test=0.4, sel_probes=[])
print(len(dft))
">>> plate_exp_dic", len(dft), len(test_list), len(train_list), ' - first item', train_list[0]

In [ ]:
len(cp.classes), cp.classes

In [ ]:
len(cp.class_to_index), cp.class_to_index

In [ ]:
root_data

In [ ]:
fname = 'crop/Plate1848/FACT - 10%SFB/Overlay_B08_site_11.tif_crop_0_ncrop_5.png'

filename = os.path.join(root_data, fname)
os.path.exists(filename), filename

In [ ]:
print(len(dft))
dft.head(2)

In [ ]:
len(cp.classes)

In [ ]:
train_list[:2]

### CellDataset_b3: Transforms for B0 or B3 - EfficientNet-B3 (300×300)

### Dataset + Loader

### Build data

In [ ]:
sel_probes = ['FACT', 'MMP1', 'Faloidina']
sel_probes = ['FACT']
sel_probes = ['Faloidina']
sel_probes = ['MMP1']
sel_probes = ['Rac1']

train_list, test_list, dft = cp.create_train_and_test_dataset(ncrop=5, perc_train=0.8, perc_test=0.2, sel_probes=sel_probes)
print(len(dft))
">>> plate_exp_dic", len(dft), len(train_list), ' - first item', train_list[0]

In [ ]:
dft

In [ ]:
len(cp.classes), cp.classes

In [ ]:
cp.perc_train, cp.perc_test

### Model (MONAI efficientnet-b3) – good for biomedical images

	def __init__(self, crop_or_segment:str, ncrop:int, sel_probes:List, classes:List, 
			     root0:str, n_determinism:int=-1, verbose:bool=False)


#### efficientnet
https://monai.readthedocs.io/en/1.3.0/_modules/monai/networks/nets/efficientnet.html

In [ ]:
classes = cp.classes

mnn = MyNN(crop_or_segment='crop', ncrop=ncrop, sel_probes=sel_probes, 
           classes=classes, root0_data=root_data, n_determinism=42, verbose=True)

In [ ]:
lr=1e-4
weight_decay=1e-4
label_smoothing=0.1
pretrained=True

In [ ]:
mnn.create_monai_EfficientNetBN_b3(lr=lr, weight_decay=weight_decay, 
                                   label_smoothing=label_smoothing, pretrained=pretrained)
# mnn.model, mnn.criterion, mnn.optimizer

### Loader

In [ ]:
ds_train = mnn.CellDataset_b3(train_list)
ds_test = mnn.CellDataset_b3(test_list)

#-- set train_loader and test_loader
mnn.set_train_and_test_dataset(ds_train, ds_test, batch_size=16, shuffle=True, num_workers=4)   

In [ ]:
len(mnn.train_loader), len(mnn.test_loader)

In [ ]:
filename, fname_model = mnn.get_model_name()
filename, fname_model

In [ ]:
#-- if found a mdel - reload train_loader and test_loader
ret, (model, train_losses, test_losses, accu_list) = mnn.read_model(verbose=True)
ret

In [ ]:
len(mnn.train_loader), len(mnn.test_loader)

In [ ]:
len(train_losses), train_losses[:5]

In [ ]:
len(test_losses), test_losses[:5]

In [ ]:
len(accu_list), accu_list[-5:]

### Use mixed precision (AMP)

AMP gives faster training + better stability:

In [ ]:
len(mnn.train_losses), len(mnn.test_losses), len(mnn.accu_list)

In [ ]:
n_epochs=20
verbose=True

mnn.train_monai_model(n_epochs=n_epochs, n_max_repeat=5, verbose=verbose)

In [ ]:
len(mnn.accu_list), np.round( [np.max(mnn.accu_list)*100, 100*mnn.maxi] ,1)

In [ ]:
mnn.plot_losses_and_accuracy(figsize=(12,5))

In [ ]:
len(mnn.train_losses)

In [ ]:
len(mnn.train_loader), len(mnn.test_loader)